# Validate Kvasir-VQA x1 splits + integrity

Checks:
- Image paths exist on disk
- No duplicate images across splits (leakage)
- Required fields present
- Build canonical `manifest_x1.parquet` for all downstream modeling


In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
import json
from pathlib import Path
import ast

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [3]:
def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    # heuristic: climb up from CWD
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").exists():
            return (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").resolve()
        if (p / "0_dataset_prep").exists() and (p / "1_dataset_analysis").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not locate Kvasir_VQA_x1 root. Set KVASIR_VQA_X1_ROOT.")

ROOT = find_kvasir_x1_root()
DATA_PREP = ROOT / "0_dataset_prep"
META_CSV = DATA_PREP / "out" / "metadata" / "metadata_enriched.csv"
MANIFEST_CSV = DATA_PREP / "out" / "manifests" / "image_manifest.csv"
OUT_MANIFEST = DATA_PREP / "out" / "manifest_x1.parquet"
OUT_REPORT = DATA_PREP / "out" / "manifest_x1_report.json"

print("ROOT:", ROOT)
print("META_CSV:", META_CSV)


ROOT: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
META_CSV: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv


In [4]:
# Load metadata
meta = pd.read_csv(META_CSV)
print("rows:", len(meta))
print("splits:", meta["split"].value_counts().to_dict())
print("columns:", list(meta.columns))


rows: 159549
splits: {'train': 143594, 'test': 15955}
columns: ['split', 'img_id', 'image_path', 'question', 'answer', 'question_type', 'answer_type', 'question_class', 'complexity', 'original', 'orig_height', 'orig_width']


In [5]:
# Required fields
required_cols = ["split", "img_id", "image_path", "question", "answer", "question_class", "complexity"]
missing = [c for c in required_cols if c not in meta.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Quick consistency checks
null_counts = meta[required_cols].isna().sum().to_dict()
print("null counts:", null_counts)


null counts: {'split': 0, 'img_id': 0, 'image_path': 0, 'question': 0, 'answer': 0, 'question_class': 0, 'complexity': 0}


In [6]:
# Resolve image paths

def resolve_image_path(p: str) -> Path:
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return Path("")
    p = str(p)
    path = Path(p)
    if path.is_absolute():
        return path
    return (ROOT / p).resolve()

meta["image_abs_path"] = meta["image_path"].apply(resolve_image_path)

# Check file existence
missing_paths = []
for p in tqdm(meta["image_abs_path"], desc="check images"):
    if not p.exists():
        missing_paths.append(str(p))

print("missing images:", len(missing_paths))
if missing_paths:
    print("first few missing:", missing_paths[:10])


check images:   0%|          | 0/159549 [00:00<?, ?it/s]

missing images: 159549
first few missing: ['/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/out/images/all/clb0kvxvm90y4074yf50vf5nq.jpg', '/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/out/images/all/cl8k2u1r71foz083278j63qnm.jpg', '/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/out/images/all/cl8k2u1qa1ekz08324rek2qcv.jpg', '/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/out/images/all/cla820gmss67b071u3h7o5k3t.jpg', '/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/out/images/all/clb0kvxvf90l4074y85pi02pq.jpg', '/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/out/images/all/cla820gn7s6nb071u382u7155.jpg', '/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/out/images/all/clb0lbwzpdpgo086ucmqx7nyu.jpg', '/workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA

In [7]:
# Check duplicate image IDs across splits (leakage)
img_split = meta.groupby("img_id")["split"].nunique()
leak_ids = img_split[img_split > 1].index.tolist()
print("img_id with >1 split:", len(leak_ids))
if leak_ids:
    print("example leaked ids:", leak_ids[:10])


img_id with >1 split: 3821
example leaked ids: ['cl8k2u1pm1dw7083203g1b7yv', 'cl8k2u1pm1dwb08325rls8okr', 'cl8k2u1pn1dwj0832bf4714ye', 'cl8k2u1pn1dwn083249fm9awa', 'cl8k2u1pn1dwr0832doqga81v', 'cl8k2u1pn1dwv0832fxargun1', 'cl8k2u1pn1dwz0832dxf2cxb5', 'cl8k2u1pn1dx3083225dq7yx3', 'cl8k2u1pn1dx708328o6h7alj', 'cl8k2u1pn1dxb0832dcxd14tf']


In [8]:
# Parse question_class into list

def parse_question_class(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        inner = s[1:-1].strip()
        if not inner:
            return []
        # handle array-like strings without commas
        if "," in inner:
            parts = [p.strip() for p in inner.split(",")]
        else:
            parts = [p.strip() for p in inner.split()]
        return [p.strip("'\"") for p in parts if p.strip("'\"")]
    return [s]

meta["question_class_list"] = meta["question_class"].apply(parse_question_class)
print(meta["question_class_list"].head().tolist())


[['abnormality_presence', 'polyp_type', 'landmark_presence'], ['procedure_type', 'polyp_type'], ['polyp_removal_status', 'text_presence', 'abnormality_location'], ['instrument_count', 'polyp_count', 'finding_count'], ['instrument_presence']]


In [9]:
# Infer transformed flag (if present), else fallback to path heuristic

def infer_transformed(df: pd.DataFrame) -> pd.Series:
    for col in ["transformed_flag", "is_transformed", "transformed", "augmented", "is_augmented"]:
        if col in df.columns:
            return df[col].astype(bool)

    # Fallback: detect keywords in path
    def _path_flag(p: str) -> bool:
        if p is None or (isinstance(p, float) and np.isnan(p)):
            return False
        s = str(p).lower()
        return any(k in s for k in ["transform", "transformed", "aug", "augmented", "noise", "noisy"])

    return df["image_path"].apply(_path_flag)

meta["is_transformed"] = infer_transformed(meta)
print(meta["is_transformed"].value_counts(dropna=False))


is_transformed
False    159549
Name: count, dtype: int64


In [10]:
# Build canonical manifest

# Keep deterministic ordering
sort_cols = ["split", "img_id", "question"]
meta_sorted = meta.sort_values(sort_cols, kind="stable").reset_index(drop=True)
meta_sorted["sample_id"] = meta_sorted.index.map(lambda i: f"x1_{i:07d}")

cols = [
    "sample_id",
    "split",
    "img_id",
    "image_path",
    "image_abs_path",
    "question",
    "answer",
    "question_class",
    "question_class_list",
    "complexity",
    "is_transformed",
]
# add optional fields if present
for extra in ["question_type", "answer_type", "original", "orig_height", "orig_width"]:
    if extra in meta_sorted.columns and extra not in cols:
        cols.append(extra)

manifest = meta_sorted[cols]

# Parquet-safe: cast paths to string
if "image_abs_path" in manifest.columns:
    manifest["image_abs_path"] = manifest["image_abs_path"].astype(str)
print("manifest columns:", list(manifest.columns))

OUT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
manifest.to_parquet(OUT_MANIFEST, index=False)
print("Wrote", OUT_MANIFEST)


manifest columns: ['sample_id', 'split', 'img_id', 'image_path', 'image_abs_path', 'question', 'answer', 'question_class', 'question_class_list', 'complexity', 'is_transformed', 'question_type', 'answer_type', 'original', 'orig_height', 'orig_width']


Wrote /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/manifest_x1.parquet


In [11]:
# Save report
report = {
    "rows": int(len(meta)),
    "splits": meta["split"].value_counts().to_dict(),
    "missing_images": int(len(missing_paths)),
    "leak_img_ids": int(len(leak_ids)),
}

with open(OUT_REPORT, "w") as f:
    json.dump(report, f, indent=2)

print("Wrote", OUT_REPORT)
print(report)


Wrote /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/manifest_x1_report.json
{'rows': 159549, 'splits': {'train': 143594, 'test': 15955}, 'missing_images': 159549, 'leak_img_ids': 3821}
